# WikiTrend Raw Inspection

This notebook inspects immutable Bronze `.gz` files using manifests, bounded samples, and streamed quality checks.

## 1. Project setup

Run this notebook from the repository root or from the `notebooks` directory.

In [ ]:
from collections import Counter
from datetime import datetime
import gzip
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pageviews'
RAW_FILENAME = re.compile(r'pageviews-(\d{8})-(\d{2})0000\.gz$')

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw directory exists: {RAW_DIR.exists()}')


## 2. Raw file manifest

The manifest is derived from the immutable `.gz` landing files. It records the date, UTC hour, path, and compressed size.

In [ ]:
manifest_rows = []
for path in sorted(RAW_DIR.rglob('*.gz')):
    match = RAW_FILENAME.search(path.name)
    if not match:
        continue
    date_value = datetime.strptime(match.group(1), '%Y%m%d').date().isoformat()
    manifest_rows.append({
        'file': str(path.relative_to(PROJECT_ROOT)),
        'date': date_value,
        'hour_utc': int(match.group(2)),
        'compressed_bytes': path.stat().st_size,
        'compressed_mb': round(path.stat().st_size / 1024**2, 2),
    })

manifest = pd.DataFrame(manifest_rows)
display(manifest.head())
display(manifest.tail())
print(f'Files: {len(manifest):,}')
print(f'Total compressed GB: {manifest.compressed_bytes.sum() / 1024**3:.2f}')
print(f'Distinct date-hours: {manifest[["date", "hour_utc"]].drop_duplicates().shape[0]:,}')

## 3. Sample raw records

Raw files have no header and contain four space-separated fields:
`project`, `page_title`, `view_count`, and `response_size`.
The reader below streams only the requested number of lines.

In [ ]:
def sample_raw(path: Path, rows: int = 10, project: str | None = None) -> pd.DataFrame:
    samples = []
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as handle:
        for line in handle:
            fields = line.rstrip('\n').split(' ')
            if len(fields) != 4:
                continue
            if project and fields[0] != project:
                continue
            samples.append({
                'project': fields[0],
                'page_title': fields[1],
                'view_count': int(fields[2]),
                'response_size': int(fields[3]),
            })
            if len(samples) >= rows:
                break
    return pd.DataFrame(samples)

raw_file = Path(manifest.iloc[0]['file'])
if not raw_file.is_absolute():
    raw_file = PROJECT_ROOT / raw_file

display(sample_raw(raw_file, rows=10))
print(f'Sampled file: {raw_file}')

In [ ]:
# Sample a selected project instead of taking the first physical rows.
display(sample_raw(raw_file, rows=10, project='en.m'))

## 4. Raw schema and first-file profiling

This scans one compressed hour to check field counts and project-code frequency. It does not scan all 83 files.

In [ ]:
field_counts = Counter()
project_counts = Counter()
with gzip.open(raw_file, 'rt', encoding='utf-8', errors='replace') as handle:
    for line in handle:
        fields = line.rstrip('\n').split(' ')
        field_counts[len(fields)] += 1
        if len(fields) == 4:
            project_counts[fields[0]] += 1

print('Field-count distribution:', field_counts)
display(pd.DataFrame(project_counts.most_common(20), columns=['project', 'rows']))

## 5. Raw quality EDA

The raw scan is sampled by default because the compressed input is several gigabytes. Set `RUN_FULL_RAW_EDA = True` for a complete scan.

In [ ]:
RUN_FULL_RAW_EDA = False
raw_paths = sorted(RAW_DIR.rglob('*.gz'))
if not RUN_FULL_RAW_EDA:
    raw_paths = raw_paths[:1]

def profile_raw_quality(paths: list[Path]):
    counters = Counter()
    projects = Counter()
    for path in paths:
        with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as handle:
            for line in handle:
                counters['raw_rows'] += 1
                fields = line.rstrip('\r\n').split(' ')
                if len(fields) != 4:
                    counters['malformed_field_count'] += 1
                    continue
                counters['four_field_rows'] += 1
                project, title, views_raw, response_raw = fields
                projects[project] += 1
                if not project:
                    counters['missing_project'] += 1
                if not title:
                    counters['missing_page_title'] += 1
                for name, raw_value in [('view_count', views_raw), ('response_size', response_raw)]:
                    try:
                        value = int(raw_value)
                    except (TypeError, ValueError):
                        counters[f'invalid_{name}'] += 1
                        continue
                    if value < 0:
                        counters[f'negative_{name}'] += 1
    metric_names = [
        'raw_rows', 'four_field_rows', 'malformed_field_count',
        'missing_project', 'missing_page_title',
        'invalid_view_count', 'negative_view_count',
        'invalid_response_size', 'negative_response_size',
    ]
    metrics = pd.DataFrame({'metric': metric_names, 'count': [counters[name] for name in metric_names]})
    project_table = pd.DataFrame(projects.most_common(20), columns=['project', 'rows'])
    return metrics, project_table

raw_quality, raw_projects = profile_raw_quality(raw_paths)
print(f'Raw files scanned: {len(raw_paths):,}')
display(raw_quality)
display(raw_projects)

## Inspection workflow

1. Check the immutable raw manifest and date/hour coverage.
2. Stream a small raw sample and confirm the four-field contract.
3. Profile malformed fields, missing values, invalid numbers, and project coverage.
4. Run `python scripts/validate_silver.py` separately after Silver processing.